# Testing


In [ ]:
import os
import re
import csv
import typing
import itertools
import json
import logging
import warnings
import IPython.display
import torch
import evaluate
import pkg_resources
import types
import pandas as pd
import sklearn as sk
import numpy as np
import estnltk, estnltk.converters, estnltk.taggers
import math
import IPython

from tqdm import tqdm
from simpletransformers.ner import NERModel, NERArgs
from bert_morph_tagger import BertMorphTagger
from morph_eval_utils import (
    MorphDiffSummarizer,
    MorphDiffFinder,
    write_formatted_diff_str_to_file,
)
from est_ud_utils import (
    load_ud_file_texts_with_corrections,
    load_ud_file_with_corrections,
)
from est_ud_morph_conv import convert_ud_layer_to_reduced_morph_layer
from bert_morph_tagger_notebook_functions import NotebookFunctions

e:\Anaconda3\envs\gpulocal\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from typing import MutableMapping, List

from estnltk import (
    Text,
    Layer,
    Tagger,
    ElementaryBaseSpan,
    EnvelopingBaseSpan,
    logger,
    Span,
)

# from estnltk_neural.taggers.embeddings.bert.bert_tokens_to_words_rewriter import BertTokens2WordsRewriter, map_words_to_bert_tokens, group_words_by_bert_tokens

from bert_tokens_to_words_rewriter import (
    BertTokens2WordsRewriter,
    map_words_to_bert_tokens,
)

In [ ]:
# Get locally imported modules from current notebook - https://stackoverflow.com/questions/40428931/package-for-listing-version-of-packages-used-in-a-jupyter-notebook - Alex P. Miller
def get_imports():

    for name, val in globals().items():
        if isinstance(val, types.ModuleType):
            # Split ensures you get root package,
            # not just imported function
            name = val.__name__.split(".")[0]

        elif isinstance(val, type):
            name = val.__module__.split(".")[0]

        # Some packages are weird and have different
        # imported names vs. system/pip names. Unfortunately,
        # there is no systematic way to get pip names from
        # a package's imported name. You'll have to add
        # exceptions to this list manually!
        poorly_named_packages = {"PIL": "Pillow", "sklearn": "scikit-learn"}
        if name in poorly_named_packages.keys():
            name = poorly_named_packages[name]

        yield name


imports = list(set(get_imports()))

# The only way I found to get the version of the root package
# from only the name of the package is to cross-check the names
# of installed packages vs. imported packages
requirements = []
for m in pkg_resources.working_set:
    if m.project_name in imports and m.project_name != "pip":
        requirements.append((m.project_name, m.version))

for r in requirements:
    print("{}=={}".format(*r))

# estnltk==1.7.3
# evaluate==0.4.2
# numpy==1.26.4
# pandas==2.2.2
# scikit-learn==1.5.1
# simpletransformers==0.70.1
# torch==2.5.0
# tqdm==4.66.5

estnltk==1.7.3
evaluate==0.4.2
numpy==1.26.4
pandas==2.2.2
scikit-learn==1.5.1
simpletransformers==0.70.1
torch==2.5.0
tqdm==4.66.5


In [ ]:
import pandas as pd
import typing
import itertools


def get_unique_labels(json_file: typing.Optional[str] = None):
    """Reads from JSON file or
    creates list of unique labels that the model must predict
    by creating all possible combinations of POS (Part Of Speech) and form.

    <i>Gathering unique labels from the enc2017 database proved to be insufficient for future model evaluation,
    because the database does not contain all possible combinations of POS and form.
    Evaluating model with UD Est-EDT test corpus proved that this problem existed.</i>

    Args:
        json_file (optional, str): Path to JSON file containing all unique labels

    Returns:
        list: List of unique labels
    """

    if json_file:
        try:
            with open(file=json_file, mode="r") as f:
                unique_labels = json.load(f)
            return unique_labels
        except:
            print("Could not read JSON file. Creating unique labels list.")

    # Separately, if one of two doesn't exist
    # pos_labels = [
    #     "A",
    #     "C",
    #     "D",
    #     "G",
    #     "H",
    #     "I",
    #     "J",
    #     "K",
    #     "N",
    #     "O",
    #     "P",
    #     "S",
    #     "U",
    #     "V",
    #     "X",
    #     "Y",
    #     "Z",
    # ]
    # form_labels = [
    #     "ab",
    #     "abl",
    #     "ad",
    #     "adt",
    #     "all",
    #     "el",
    #     "es",
    #     "g",
    #     "ill",
    #     "in",
    #     "kom",
    #     "n",
    #     "p",
    #     "pl",
    #     "sg",
    #     "ter",
    #     "tr",
    #     "b",
    #     "d",
    #     "da",
    #     "des",
    #     "ge",
    #     "gem",
    #     "gu",
    #     "ks",
    #     "ksid",
    #     "ksime",
    #     "ksin",
    #     "ksite",
    #     "ma",
    #     "maks",
    #     "mas",
    #     "mast",
    #     "mata",
    #     "me",
    #     "n",
    #     "neg",
    #     "neg ge",
    #     "neg gem",
    #     "neg gu",
    #     "neg ks",
    #     "neg me",
    #     "neg nud",
    #     "neg nuks",
    #     "neg o",
    #     "neg vat",
    #     "neg tud",
    #     "nud",
    #     "nuks",
    #     "nuksid",
    #     "nuksime",
    #     "nuksin",
    #     "nuksite",
    #     "nuvat",
    #     "o",
    #     "s",
    #     "sid",
    #     "sime",
    #     "sin",
    #     "site",
    #     "ta",
    #     "tagu",
    #     "taks",
    #     "takse",
    #     "tama",
    #     "tav",
    #     "tavat",
    #     "te",
    #     "ti",
    #     "tud",
    #     "tuks",
    #     "tuvat",
    #     "v",
    #     "vad",
    #     "vat",
    # ]

    pos_mutable = ["A", "C", "H", "N", "O", "P", "S", "U", "X", "Y"]
    pos_immutable = ["D", "G", "I", "J", "K", "X", "Z"]
    pos_verb = ["V"]
    form_mutable = [
        "ab",
        "abl",
        "ad",
        "adt",
        "all",
        "el",
        "es",
        "g",
        "ill",
        "in",
        "kom",
        "n",
        "p",
        "ter",
        "tr",
    ]
    form_mutable_count = ["pl", "sg"]
    form_verb = [
        "b",
        "d",
        "da",
        "des",
        "ge",
        "gem",
        "gu",
        "ks",
        "ksid",
        "ksime",
        "ksin",
        "ksite",
        "ma",
        "maks",
        "mas",
        "mast",
        "mata",
        "me",
        "n",
        "neg",
        "neg ge",
        "neg gem",
        "neg gu",
        "neg ks",
        "neg me",
        "neg nud",
        "neg nuks",
        "neg o",
        "neg vat",
        "neg tud",
        "nud",
        "nuks",
        "nuksid",
        "nuksime",
        "nuksin",
        "nuksite",
        "nuvat",
        "o",
        "s",
        "sid",
        "sime",
        "sin",
        "site",
        "ta",
        "tagu",
        "taks",
        "takse",
        "tama",
        "tav",
        "tavat",
        "te",
        "ti",
        "tud",
        "tuks",
        "tuvat",
        "v",
        "vad",
        "vat",
    ]

    # Combinations of only form
    only_form = (
        form_mutable
        + form_verb
        + list(itertools.product(form_mutable_count, form_mutable))
    )

    # Combinations of only pos
    only_pos = pos_mutable + pos_immutable + pos_verb

    # Combinations of mutable pos and their forms
    mutable_combinations = (
        list(
            itertools.product(
                pos_mutable, form_mutable
            )  # Non-verb combinations without count
        )
        + list(
            itertools.product(
                pos_mutable, itertools.product(form_mutable_count, form_mutable)
            )  # Non-verb combinations with count
        )
        + list(
            itertools.product(pos_verb, form_verb)  # Verb combinations
        )
    )

    # Combinations of verb pos and their forms
    verb_combinations = list(itertools.product(pos_verb, form_verb))

    # Labels
    # TODO: Check if all combinations of labels are represented
    mutable_labels = list()
    immutable_labels = [("", pos) for pos in pos_immutable]
    verb_labels = list(itertools.product(form_labels_verb, pos_label_verb))

    # Connect count and form in mutables
    for form, pos in noun_labels_nested:
        noun_labels.append((form[0] + " " + form[1], pos))

    pos_label_only = [("", pos) for pos in pos_labels]
    form_label_only = [(form, "") for form in form_labels]
    unknown_form_labels = [
        ("?", pos) for pos in pos_labels
    ]  # form '?' comes from enc2017 corpus after tagging

    unique_labels = (
        pos_label_only
        + form_label_only
        + noun_labels
        + verb_labels
        + unknown_form_labels
        + form_pos_labels
        + ["?"]
    )  # '?' for labels unknown to Vabamorf

    unique_labels_df = pd.DataFrame(unique_labels, columns=["form", "pos"])
    unique_labels_df["labels"] = unique_labels_df.apply(
        lambda row: (
            row["form"] + " " + row["pos"]
            if row["form"] and row["pos"]
            else row["form"] or row["pos"]
        ),
        axis=1,
    )
    # Drop form and pos columns and keep only labels column
    unique_labels_df.drop(labels=["form", "pos"], axis=1)
    print("List of unique labels created")
    return unique_labels_df["labels"].tolist()

Initializing the model<!-- Mudeli ülesehitamine -->


In [ ]:
def initialize_model(model_name, unique_labels, is_progress_bars=False):
    # Set up logging
    logger = logging.getLogger("simpletransformers.ner.ner_model")
    logger.setLevel(logging.ERROR)

    # Suppress specific warnings
    # warnings.filterwarnings("ignore", category=FutureWarning) # For warning message "FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated."
    warnings.filterwarnings(
        "ignore", category=UserWarning
    )  # For warnings like "UserWarning: <tag> seems not to be NE tag."

    # Configurations
    model_args = NERArgs()
    model_args.train_batch_size = 8
    model_args.evaluate_during_training = False
    model_args.learning_rate = 5e-5
    model_args.num_train_epochs = 10
    model_args.use_early_stopping = True
    model_args.use_cuda = torch.cuda.is_available()  # Use GPU if available
    model_args.save_eval_checkpoints = False
    model_args.save_model_every_epoch = False  # Takes a lot of storage space
    model_args.save_steps = -1
    model_args.overwrite_output_dir = True
    model_args.cache_dir = model_name + "/cache"
    model_args.best_model_dir = model_name + "/best_model"
    model_args.output_dir = model_name
    model_args.use_multiprocessing = False
    model_args.silent = is_progress_bars

    # Initialization
    model = NERModel("camembert", model_name, args=model_args, labels=unique_labels)
    return model

Function to predict tags to words


In [ ]:
def predict_tags(model, sentences):

    if isinstance(sentences, str):
        text = estnltk.Text(sentences)
        text.tag_layer("morph_analysis")
        text.morph_analysis
        sentences = [s.text for s in text.sentences]
        predictions, raw_outputs = model.predict(sentences, split_on_space=False)

    elif isinstance(sentences, list) and isinstance(sentences[0], list):
        predictions, raw_outputs = model.predict(sentences, split_on_space=False)

    else:
        raise TypeError(
            f"Input is in wrong format. Possible formats are str or list of lists. Your input is {type(sentences)}"
        )

    return predictions, raw_outputs

In [ ]:
with open("unique_labels.json", "r") as f:
    unique_labels = json.load(f)

model = initialize_model("NER_mudel", unique_labels)

In [7]:
model.tokenizer.vocab_file

'NER_mudel\\sentencepiece.bpe.model'

### Debugging


In [ ]:
in_dir = "_plain_texts_json"
comparison_data = pd.read_csv("comparison_data.csv", keep_default_na=False)
jsons = comparison_data["source"].unique().tolist()
morph_tagger = BertMorphTagger(
    "./NER_mudel_v2/", get_top_n_predictions=1, token_level=False
)

In [ ]:
morph_diff_finder = MorphDiffFinder(
    "morph_analysis",
    "bert_morph_tagging",
    diff_attribs=["partofspeech", "form"],
    focus_attribs=["partofspeech", "form"],
)
morph_diff_summarizer = MorphDiffSummarizer("morph_analysis", "bert_morph_tagging")

In [ ]:
# input_str = 'Aga muidugi, loodrid, kes soovivad stabiilset ja mugavat, 9.00-17.00 elu, on muidugi hea rääkida, et oi, kus ma siin nüüd rügan.'
# input_str = 'Ning laste kingitus: puust juuksehari (millist ma olin ammu endale soovinud), tulbid ja karp Rafaellot.'
input_str = "( nt kõrgenenud keskkõrge e puhul : � , � , î )"

In [8]:
text_obj = estnltk.Text(input_str)
# text_obj = estnltk.converters.json_to_text(file=os.path.join('./bert_morph_texts_json/', jsons[931]))
# text_obj = estnltk.converters.json_to_text(file=os.path.join('./_vabamorf_morph_texts_json/', jsons[10]))
# text_obj = estnltk.converters.json_to_text(file=os.path.join('./_bert_morph_texts_json/', jsons[27]))
# text_obj = estnltk.converters.json_to_text(file=os.path.join('./_vabamorf_morph_texts_json/', jsons[187]))
# text_obj = estnltk.converters.json_to_text(file=os.path.join('./_vabamorf_morph_texts_json/', jsons[186]))
# text_obj = estnltk.converters.json_to_text(file=os.path.join(in_dir, 'nc_17153_741265.json'))
text_obj.tag_layer("morph_analysis");

In [9]:
text_obj

Text(text='( nt kõrgenenud keskkõrge e puhul : � , � , î )')

In [10]:
# Apply the tagger to the text object
text_obj.add_layer(morph_tagger.make_layer(text_obj))

c:\Users\Admin\OneDrive\TU\Praktika\EstNLTK\morf_yhestaja\bert_tokens_to_words_rewriter.py:154: UserWarning: (!) No matching words span for bert token Span(' ', [{'bert_tokens': '▁', 'form': '?', 'partofspeech': '', 'probability': 0.80297}]).
  warnings.warn(f"(!) No matching {words_layer.name} span for bert token {bert_tokens_layer[i]}.")


In [11]:
text_obj.bert_morph_tagging

Layer(name='bert_morph_tagging', attributes=('bert_tokens', 'form', 'partofspeech', 'probability'), spans=SL[Span('(', [{'bert_tokens': ['▁('], 'form': '', 'partofspeech': 'Z', 'probability': 0.99999}]),
Span('nt', [{'bert_tokens': ['▁nt'], 'form': '', 'partofspeech': 'Y', 'probability': 0.99042}]),
Span('kõrgenenud', [{'bert_tokens': ['▁kõr', 'genenud'], 'form': '', 'partofspeech': 'A', 'probability': 0.99978}]),
Span('keskkõrge', [{'bert_tokens': ['▁keskk', 'õr', 'ge'], 'form': 'sg g', 'partofspeech': 'S', 'probability': 0.99644}]),
Span('e', [{'bert_tokens': ['▁e'], 'form': '', 'partofspeech': 'Y', 'probability': 0.99819}]),
Span('puhul', [{'bert_tokens': ['▁puhul'], 'form': '', 'partofspeech': 'K', 'probability': 0.99939}]),
Span(':', [{'bert_tokens': ['▁:'], 'form': '', 'partofspeech': 'Z', 'probability': 0.99998}]),
Span(',', [{'bert_tokens': ['▁,'], 'form': '', 'partofspeech': 'Z', 'probability': 0.99999}]),
Span(',', [{'bert_tokens': ['▁,'], 'form': '', 'partofspeech': 'Z', 'probability': 0.99999}]),
Span('î', [{'bert_tokens': ['î'], 'form': '?', 'partofspeech': '', 'probability': 0.86856}]),
Span(')', [{'bert_tokens': ['▁)'], 'form': '', 'partofspeech': 'Z', 'probability': 0.99999}])])

In [12]:
text_obj.words

Layer(name='words', attributes=('normalized_form',), spans=SL[Span('(', [{'normalized_form': None}]),
Span('nt', [{'normalized_form': 'nt'}]),
Span('kõrgenenud', [{'normalized_form': None}]),
Span('keskkõrge', [{'normalized_form': None}]),
Span('e', [{'normalized_form': None}]),
Span('puhul', [{'normalized_form': None}]),
Span(':', [{'normalized_form': None}]),
Span('�', [{'normalized_form': None}]),
Span(',', [{'normalized_form': None}]),
Span('�', [{'normalized_form': None}]),
Span(',', [{'normalized_form': None}]),
Span('î', [{'normalized_form': None}]),
Span(')', [{'normalized_form': None}])])

In [57]:
# display(text_obj.bert_morph_tagging[50:55])
# display(text_obj.morph_analysis[50:55])

In [ ]:
morph_diff_layer, formatted_diffs_str, total_diff_gaps = (
    morph_diff_finder.find_difference(text_obj, fname=jsons[0], text_cat="wiki")
)  # text_obj.meta['texttype']

In [ ]:
fpath = os.path.join(
    "differences_test",
    f"_{jsons[0]}__ann_diffs_{datetime.now().strftime('%Y-%m-%dT%H%M%S')}.txt",
)
write_formatted_diff_str_to_file(fpath, formatted_diffs_str)

In [ ]:
morph_diff_summarizer.record_from_diff_layer(
    "morph_analysis", morph_diff_layer, "wiki"
)  # text_obj.meta['texttype'] <- None asemele tuleb hiljem tekstiliik võetud meta-andmetest

In [ ]:
morph_diff_summarizer.get_diffs_summary_output(show_doc_count=True)

'morph_analysis\n wiki  | #docs: 1 | modified spans: 2 / 20 (0.1000) | annotations ratio: 38 / 43 (0.8837) | only in morph_analysis: 4 (9.3023%) | only in bert_morph_tagging: 1 (2.3256%) \n\n'

In [ ]:
text_obj.meta["texttype"]

'blogs_and_forums'

In [13]:
# print(morph_diff_finder._bert_vabamorf_tags_layer)
# print(morph_diff_finder.new_layer)
# print(morph_diff_finder.old_layer)
# print(morph_diff_finder.old_layer + morph_diff_finder._redux_layer_suffix)
# print(morph_diff_finder.old_layer + morph_diff_finder._flat_layer_suffix)
# print(morph_diff_finder._bert_vabamorf_tags_layer + morph_diff_finder._redux_layer_suffix)
# print(morph_diff_finder._bert_vabamorf_tags_layer + morph_diff_finder._flat_layer_suffix)

In [ ]:
new_layer = Layer(
    name="bert_morph_tags",
    attributes=["bert_tokens", "form", "partofspeech"],
    text_object=text_obj,
    parent="words",
    ambiguous=True,
)
for sp in text_obj.bert_morph_tagging.spans:
    span = sp.base_span
    tokens = sp.bert_tokens[0]
    label = sp.morph_labels[0].split("_")
    pos = ""
    form = ""
    if len(label) == 1:  # Only one feature exissts (form or partofspeech)
        if label[0].isupper():  # partofspeech is uppercased
            pos = sp.morph_labels[0].split("_")[0]
        else:
            form = sp.morph_labels[0].split("_")[0]
    else:  # Both features exist
        form = sp.morph_labels[0].split("_")[0]
        pos = sp.morph_labels[0].split("_")[1]

    annotation = {"bert_tokens": tokens, "form": form, "partofspeech": pos}
    new_layer.add_annotation(span, annotation)
text_obj.add_layer(new_layer)

In [ ]:
def create_redux_morph_layer(text_obj, morph_layer, output_layer, add_layer=True):
    """Creates a reduced version of morph_analysis layer which only contains attributes 'partofspeech' and 'form'.
    The reduced morph layer is required for comparing Vabamorf's annotations against Bert annotations
    via DiffTagger (because DiffTagger assumes that comparable layers have common attributes).
    """
    assert isinstance(text_obj, Text)
    assert morph_layer in text_obj.layers, "(!) Layer {!r} missing from: {!r}".format(
        morph_layer, text_obj.layers.keys()
    )
    redux_layer = Layer(
        name=output_layer,
        attributes=("partofspeech", "form"),
        text_object=text_obj,
        ambiguous=True,
    )
    for morph_word in text_obj[morph_layer]:
        for ann in morph_word.annotations:
            assert "partofspeech" in ann.keys(), (
                "(!) Attribute {!r} missing from: {!r}".format("pos", ann)
            )
            assert "form" in ann.keys(), "(!) Attribute {!r} missing from: {!r}".format(
                "form", ann
            )
            postag = ann["partofspeech"]
            form = ann["form"]
            redux_layer.add_annotation(
                morph_word.base_span, {"partofspeech": postag, "form": form}
            )
    if add_layer:
        text_obj.add_layer(redux_layer)
    return redux_layer

In [ ]:
create_redux_morph_layer(
    text_obj, "morph_analysis", "morph_analysis_redux", add_layer=True
)
create_redux_morph_layer(
    text_obj, "bert_morph_tags", "bert_morph_tags_redux", add_layer=True
)
morph_diff_finder = MorphDiffFinder(
    "morph_analysis_redux",
    "bert_morph_tags_redux",
    diff_attribs=["partofspeech", "form"],
    focus_attribs=["partofspeech", "form"],
)

In [14]:
morph_diff_finder.find_difference(text_obj, fname=jsons[0])

(Layer(name='morph_diff_layer', attributes=('input_layer_name', 'span_status', 'partofspeech', 'form'), spans=SL[Span('Jüri', [{'input_layer_name': 'morph_analysis_flat', 'span_status': 'modified', 'partofspeech': 'H', 'form': 'sg n'}, {'input_layer_name': 'bert_vabamorf_tags_flat', 'span_status': 'modified', 'partofspeech': 'H', 'form': 'sg g'}]),
 Span('küpsetatud', [{'input_layer_name': 'morph_analysis_flat', 'span_status': 'modified', 'partofspeech': 'A', 'form': 'sg n'}, {'input_layer_name': 'morph_analysis_flat', 'span_status': 'modified', 'partofspeech': 'A', 'form': 'pl n'}, {'input_layer_name': 'morph_analysis_flat', 'span_status': 'modified', 'partofspeech': 'V', 'form': 'tud'}]),
 Span('soovinud', [{'input_layer_name': 'morph_analysis_flat', 'span_status': 'modified', 'partofspeech': 'A', 'form': 'pl n'}, {'input_layer_name': 'morph_analysis_flat', 'span_status': 'modified', 'partofspeech': 'A', 'form': 'sg n'}, {'input_layer_name': 'morph_analysis_flat', 'span_status': 'mod

In [13]:
text_obj.words;

In [14]:
print(len(vm_spans), len(bert_spans))

35349 34331


In [39]:
text_obj.sentences[29]

text
"Varasemates transkribeeritud tekstides võib leida nende vokaalide kolmesugust märkimist ( nt kõrgenenud keskkõrge e puhul : � , � , î ) ."


In [ ]:
bert_spans = text_obj.bert_morph_tagging.spans
vm_spans = text_obj.words.spans
shift = 0
print(len(vm_spans), len(bert_spans))
if len(vm_spans) < len(bert_spans):
    for i in range(len(vm_spans)):
        if vm_spans[i - shift].text != bert_spans[i].text:
            print(i, i + shift, vm_spans[i].text, bert_spans[i + shift].text)
            shift += 1
            for n in range(i - 10, i + 10 + 1):
                print(n, vm_spans[n].text, bert_spans[n].text)
            print()
        if shift > 10:
            break
else:
    for i in range(len(bert_spans)):
        if vm_spans[i].text != bert_spans[i - shift].text:
            print(i, i + shift, vm_spans[i].text, bert_spans[i + shift].text)
            shift += 1
            for n in range(i - 10, i + 10 + 1):
                print(n, vm_spans[n].text, bert_spans[n].text)
            print()
        if shift > 10:
            break

35349 34331
539 539 � ,
529 vokaalide vokaalide
530 kolmesugust kolmesugust
531 märkimist märkimist
532 ( (
533 nt nt
534 kõrgenenud kõrgenenud
535 keskkõrge keskkõrge
536 e e
537 puhul puhul
538 : :
539 � ,
540 , ,
541 � î
542 , )
543 î .
544 ) Transkribeerimine
545 . võib
546 Transkribeerimine ju
547 võib mingil
548 ju määral
549 mingil anda

541 542 � )
531 märkimist märkimist
532 ( (
533 nt nt
534 kõrgenenud kõrgenenud
535 keskkõrge keskkõrge
536 e e
537 puhul puhul
538 : :
539 � ,
540 , ,
541 � î
542 , )
543 î .
544 ) Transkribeerimine
545 . võib
546 Transkribeerimine ju
547 võib mingil
548 ju määral
549 mingil anda
550 määral edasi
551 anda nende

1452 1454 � ,
1442 i ü
1443 , ,
1444 ü e
1445 , ,
1446 e ö
1447 , ,
1448 ö ä
1449 , ,
1450 ä ,
1451 , u
1452 � ,
1453 , o
1454 u ,
1455 , a
1456 o ]
1457 , ning
1458 a kümme
1459 ] vokaali
1460 ning Võru
1461 kümme murdes
1462 vokaali [

1476 1479 � a
1466 i ,
1467 , e
1468 ü ,
1469 , ö
1470 e ,
1471 , ä
1472 ö ,
1473 , ,
1474 ä ,
1475 

In [ ]:
morph_tagger.bert_tokenizer(input_str, return_tensors="pt")["input_ids"]

tensor([[    5,   156,  3487,   492, 27200,  7286,  2890,    78,    15,  1098,
           262,    36,    36, 39638, 39901,   300,     6]])

In [13]:
morph_tagger.bert_tokenizer(input_str)

{'input_ids': [5, 156, 3487, 492, 27200, 7286, 2890, 78, 15, 1098, 262, 36, 36, 39638, 39901, 300, 6], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [14]:
morph_tagger._tokenize_with_bert(input_str)

([(None, None, '<s>'),
  (0, 1, '▁('),
  (1, 4, '▁nt'),
  (4, 8, '▁kõr'),
  (8, 15, 'genenud'),
  (15, 21, '▁keskk'),
  (21, 23, 'õr'),
  (23, 25, 'ge'),
  (25, 27, '▁e'),
  (27, 33, '▁puhul'),
  (33, 35, '▁:'),
  (37, 39, '▁,'),
  (41, 43, '▁,'),
  (43, 44, '▁'),
  (44, 45, 'î'),
  (45, 47, '▁)'),
  (None, None, '</s>')],
 {'input_ids': [5, 156, 3487, 492, 27200, 7286, 2890, 78, 15, 1098, 262, 36, 36, 39638, 39901, 300, 6], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]})

In [ ]:
for token_data in morph_tagger._get_bert_morph_tagging_label_predictions(input_str, 3):
    start, end = token_data["token"][0], token_data["token"][1]
    bert_tokens = token_data["token"][2]
    if start is None or end is None:
        continue  # Ignore sentence start and end tokens (<s>, </s>)
    all_labels = [pred["label"] for pred in token_data["predictions"]]
    all_probabilities = [pred["probability"] for pred in token_data["predictions"]]
    for label, prob in zip(all_labels, all_probabilities):
        print(label[0], label[1], prob)

 J 0.99991
o V 0.0
 D 0.0
pl g S 0.9996
sg g S 6e-05
pl g A 3e-05
sg n S 0.99978
sg in S 3e-05
sg p S 1e-05
 Z 0.99992
 J 2e-05
b V 0.0
sg el S 0.99941
sg p S 0.00012
sg el H 4e-05
sg n S 0.99975
sg in S 4e-05
sg p S 3e-05
sg n S 0.99975
sg in S 4e-05
sg p S 2e-05
 Z 0.99987
 J 5e-05
 A 0.0
sg p P 0.99783
sg el P 0.00098
sg p A 0.0001
sg p P 0.99854
sg el P 0.00043
sg p A 0.0001
sg n P 0.99983
pl n P 1e-05
sg g P 1e-05
sin V 0.99878
sime V 9e-05
n V 8e-05
 D 0.99989
 A 1e-05
sg g S 0.0
sg all P 0.9998
pl all P 2e-05
sg all H 1e-05
nud V 0.99973
pl n S 2e-05
 A 2e-05
 Z 0.99995
 J 0.0
 A 0.0
pl n S 0.99949
pl n A 0.00012
pl n H 5e-05
pl n S 0.99952
pl n A 0.00012
pl n H 4e-05
 J 0.99991
o V 0.0
 D 0.0
sg n S 0.99976
sg in S 2e-05
pl g S 2e-05
sg p H 0.78436
sg n H 0.1822
pl n H 0.00609
sg p H 0.88837
sg n H 0.08277
pl n H 0.00468
sg p H 0.9153
sg n H 0.05623
pl n H 0.00461
sg p H 0.91316
sg n H 0.05442
pl n H 0.00383
 Z 0.99996
 J 0.0
sg n H 0.0


In [ ]:
def _tokenize_with_bert(self, text, include_spanless=True):
    """
    Tokenizes input string with Bert's tokenizer and returns a list of token spans.
    Each token span is a triple (start, end, token).
    If include_spanless==True (default), then Bert's special "spanless" tokens
    (e.g. [CLS], [SEP]) will also be included with their respective start/end indexes
    set to None.
    """
    tokens = []
    batch_encoding = self.bert_tokenizer(text)

    # Iterate over tokens and their character spans
    for token_id, token in enumerate(batch_encoding.tokens()):
        char_span = batch_encoding.token_to_chars(token_id)

        # Skip any spanless tokens like [CLS], [SEP] unless include_spanless is set to True
        if char_span is None and include_spanless:
            tokens.append((None, None, token))
            continue
        elif char_span is None:
            continue

        # Initialize the current token list to collect non-punctuation parts
        start, end = char_span.start, char_span.end
        clean_token = []

        # Iterate over each character in the token to separate punctuation
        for i, char in enumerate(token):
            if (
                char.isalnum() or char == "▁"
            ):  # Alphanumeric or BERT's special start marker
                clean_token.append(char)
            else:
                # If clean_token has content, add it as a token with the current span
                if clean_token:
                    tokens.append(
                        (start, start + len(clean_token), "".join(clean_token))
                    )
                    clean_token = []

                # Add the punctuation as a separate token with its span
                tokens.append((char, start + i, start + i + 1))

        # Add any remaining non-punctuation part as a token
        if clean_token:
            tokens.append((start, start + len(clean_token), "".join(clean_token)))

    return tokens

In [ ]:
# Use the refactored _tokenize_with_bert method to get the correct tokens and their spans
tokens_with_spans = morph_tagger._tokenize_with_bert(input_str)

# Extract only the tokens for encoding
clean_tokens = [str(token) for _, _, token in tokens_with_spans]

# Tokenize the input string with BERT's tokenizer for obtaining token indices
token_indexes = morph_tagger.bert_tokenizer.convert_tokens_to_ids(clean_tokens)
print(token_indexes)
token_indexes = torch.tensor([token_indexes])  # Prepare for model input

# Check if input exceeds max sequence length
max_seq_length = morph_tagger.bert_tokenizer.model_max_length
if token_indexes.size(1) > max_seq_length:
    raise ValueError(
        f"Input length exceeds the model's max_seq_length of {max_seq_length} tokens"
    )

# Get model predictions (logits)
with torch.no_grad():
    output = morph_tagger.bert_morph_tagging(token_indexes)

logits = output.logits.squeeze(0)  # Shape: [sequence_length, num_labels]
probs = torch.softmax(logits, dim=-1)  # Convert logits to probabilities

# Assign predictions to tokens
top_n_predictions = []
for i, (start, end, token) in enumerate(tokens_with_spans):
    if start is None or end is None or token == "▁":
        continue  # Ignore sentence start and end tokens (<s>, </s>) and word start sign (▁)

    # Get probabilities for the current token
    token_probs = probs[i]
    top_n_indices = torch.topk(
        token_probs, morph_tagger.get_top_n_predictions
    ).indices  # Top N label indices
    top_n_labels = [
        morph_tagger.id2label[idx.item()] for idx in top_n_indices
    ]  # Convert indices to labels
    top_n_probs = [
        round(token_probs[idx].item(), 5) for idx in top_n_indices
    ]  # Get probabilities for top N labels

    # Add the predictions to the output
    top_n_predictions.append(
        {
            "token": (start, end, token),
            "predictions": [
                {"label": label, "probability": prob}
                for label, prob in zip(top_n_labels, top_n_probs)
            ],
        }
    )

NameError: name 'morph_tagger' is not defined

In [7]:
top_n_predictions

NameError: name 'top_n_predictions' is not defined

In [8]:
# len(_split_sentence_into_smaller_chunks(text_obj.text)[0][2])

In [9]:
text_obj.bert_morph_tagging.bert_tokens

NameError: name 'text_obj' is not defined

In [10]:
# predictions, raw_outputs = predict_tags(model, input_str)

In [11]:
# raw_outputs[0][0].keys()

In [31]:
text_obj.words[0].text

'Ning'

In [ ]:
sentences_layer = text_obj.sentences

morph_layer = Layer(
    name=morph_tagger.output_layer,
    attributes=morph_tagger.output_attributes,
    text_object=text_obj,
    parent="words",
    ambiguous=True,
)

In [ ]:
def rewriter_decorator(text_obj, word_index, span):
    """
    Decorator function for <code>BertTokens2WordsRewriter</code>. \n
    Aggregates the <code>morph_labels</code> and <code>probabilities</code> from <code>shared_bert_tokens</code>, finds the most
    common top-1 label, and retrieves the top N labels and their probabilities from the
    first token that contains this top-1 label.

    Args:
        text_obj: EstNLTK Text object.
        words_index: Index of the word in <code>words</code> layer.
        span: Span

    Returns:
        dict: Annotations with the top N labels and probabilities for the word/phrase.
    """

    # Step 1: Find the most frequent top-1 label across all tokens
    top_1_label_counts = collections.Counter()

    for sp in span:
        top_1_label = sp["morph_labels"][0]  # Get top-1 label
        # 'morph_labels': ['label', 'label', ...], first [0] to get the list of labels, second [0] to get the first element
        top_1_label_counts[top_1_label] += 1  # Count occurrences of each top-1 label

    # Identify the most frequent top-1 label
    most_frequent_label = top_1_label_counts.most_common(1)[0][0]
    annotations = list()

    # Step 2: Find the first token that has this most frequent top-1 label
    for sp in span:
        if most_frequent_label in sp["morph_labels"]:
            # Extract the top N labels and their probabilities starting from this label
            labels = sp["morph_labels"]
            probabilities = sp["probabilities"]

            assert len(labels) == len(probabilities)

            for label, prob in zip(labels, probabilities):
                annotation = {
                    "bert_tokens": [sp["bert_tokens"][0] for sp in span],
                    "morph_labels": label,
                    "probabilities": prob,
                }
                annotations.append(annotation)

            # Return the final annotation
            return annotations

    # Fallback if no label found (shouldn't happen)
    raise RuntimeError(f"Could not find a token with this label: {most_frequent_label}")

In [ ]:
for k, sentence in enumerate(sentences_layer):
    sent_start = sentence.start
    sent_text = sentence.enclosing_text
    # Apply batch processing: split larger input sentence into smaller chunks and
    # process chunk by chunk
    sent_chunks, sent_chunk_indexes = _split_sentence_into_smaller_chunks(sent_text)
    for sent_chunk, (chunk_start, chunk_end) in zip(sent_chunks, sent_chunk_indexes):
        print(f"<s>{sent_chunk}</s>")
        # Get predictions for the sentence
        top_n_predictions = morph_tagger._get_bert_morph_tagging_label_predictions(
            sent_chunk, morph_tagger.get_top_n_predictions
        )

        # Collect token level annotations (a label for each token)
        for token_data in top_n_predictions:
            start, end = token_data["token"][0], token_data["token"][1]
            bert_tokens = token_data["token"][2]
            if start is None or end is None or bert_tokens == "▁":
                continue  # Ignore sentence start and end tokens (<s>, </s>) and word start sign (▁)
            all_labels = [pred["label"] for pred in token_data["predictions"]]
            all_probabilities = [
                pred["probability"] for pred in token_data["predictions"]
            ]
            token_span = (
                sent_start + chunk_start + start,
                sent_start + chunk_start + end,
            )
            print(sent_start, chunk_start, start, end)
            print(token_data)
            print(token_span)
            for label, prob in zip(all_labels, all_probabilities):
                annotation = {
                    "bert_tokens": bert_tokens,
                    "morph_labels": label,
                    "probabilities": prob,
                }
                morph_layer.add_annotation(token_span, **annotation)
            # annotation = {'bert_tokens' : bert_tokens, 'morph_label' : predicted_label }
            # token_level_annotations.append([token_span, annotation, single_label])

# Add annotations
# if morph_tagger.token_level:
#     # Return token level annotations
#     morph_layer

# else:
#     # Aggregate tokens back into words/phrases
#     # Use BertTokens2WordsRewriter to convert BERT tokens to words
#     rewriter = BertTokens2WordsRewriter(
#         bert_tokens_layer=morph_tagger.output_layer,
#         input_words_layer=morph_tagger.words_layer,
#         output_attributes=morph_tagger.output_attributes,
#         output_layer=morph_tagger.output_layer,
#         decorator=rewriter_decorator)

# #     # Rewrite to align BERT tokens with words
#     morph_layer = rewriter.make_layer(text_obj, layers={morph_layer.name: morph_layer})

<s>Koju jõudes ootas ees üllatus - Jüri küpsetatud imemaitsev shokolaadikook.</s>
0 0 0 4
{'token': (0, 4, '▁Koju'), 'predictions': [{'label': 'adt_S', 'probability': 0.93019}]}
(0, 4)
0 0 4 11
{'token': (4, 11, '▁jõudes'), 'predictions': [{'label': 'des_V', 'probability': 0.99942}]}
(4, 11)
0 0 11 17
{'token': (11, 17, '▁ootas'), 'predictions': [{'label': 's_V', 'probability': 0.99991}]}
(11, 17)
0 0 17 21
{'token': (17, 21, '▁ees'), 'predictions': [{'label': 'D', 'probability': 0.99948}]}
(17, 21)
0 0 21 29
{'token': (21, 29, '▁üllatus'), 'predictions': [{'label': 'sg n_S', 'probability': 0.99972}]}
(21, 29)
0 0 29 31
{'token': (29, 31, '▁-'), 'predictions': [{'label': 'Z', 'probability': 0.99996}]}
(29, 31)
0 0 31 36
{'token': (31, 36, '▁Jüri'), 'predictions': [{'label': 'sg g_H', 'probability': 0.80241}]}
(31, 36)
0 0 36 47
{'token': (36, 47, '▁küpsetatud'), 'predictions': [{'label': 'A', 'probability': 0.9985}]}
(36, 47)
0 0 47 51
{'token': (47, 51, '▁ime'), 'predictions': [{'labe

In [21]:
map_words_to_bert_tokens(text_obj.words, morph_layer)

{0: [Span('(', [{'bert_tokens': '▁(', 'morph_labels': 'Z', 'probabilities': 0.99991}])],
 1: [Span('Ameerika', [{'bert_tokens': 'Ameerika', 'morph_labels': 'sg g_H', 'probabilities': 0.99858}])],
 2: [Span(' teadlased', [{'bert_tokens': '▁teadlased', 'morph_labels': 'pl n_S', 'probabilities': 0.99957}])],
 3: [Span(' on', [{'bert_tokens': '▁on', 'morph_labels': 'b_V', 'probabilities': 0.99992}])],
 4: [Span(' kindlaks', [{'bert_tokens': '▁kindlaks', 'morph_labels': 'sg tr_A', 'probabilities': 0.99904}])],
 5: [Span(' teinud', [{'bert_tokens': '▁teinud', 'morph_labels': 'nud_V', 'probabilities': 0.99971}])],
 6: [Span(',', [{'bert_tokens': ',', 'morph_labels': 'Z', 'probabilities': 0.99996}])],
 7: [Span(' et', [{'bert_tokens': '▁et', 'morph_labels': 'J', 'probabilities': 0.99989}])],
 8: [Span(' 99%', [{'bert_tokens': '▁99%', 'morph_labels': '?_N', 'probabilities': 0.99934}])],
 9: [Span(' blog', [{'bert_tokens': '▁blog', 'morph_labels': 'pl in_S', 'probabilities': 0.99847}]),
  Span('

In [ ]:
mapping = map_words_to_bert_tokens(text_obj.words, morph_layer)
for k, v in mapping.items():
    # print([sp['bert_tokens'][0] for sp in v])
    print(
        k,
        text_obj.words[k].text,
        text_obj.words[k].base_span,
        [sp.annotations[0].bert_tokens for sp in v],
    )
    # rewriter_decorator(text_obj, k, v)
print(len(mapping), len(text_obj.words))

0 ( ElementaryBaseSpan(0, 1) ['▁(']
1 Ameerika ElementaryBaseSpan(1, 9) ['Ameerika']
2 teadlased ElementaryBaseSpan(10, 19) ['▁teadlased']
3 on ElementaryBaseSpan(20, 22) ['▁on']
4 kindlaks ElementaryBaseSpan(23, 31) ['▁kindlaks']
5 teinud ElementaryBaseSpan(32, 38) ['▁teinud']
6 , ElementaryBaseSpan(38, 39) [',']
7 et ElementaryBaseSpan(40, 42) ['▁et']
8 99% ElementaryBaseSpan(43, 46) ['▁99%']
9 blogides ElementaryBaseSpan(47, 55) ['▁blog', 'ides']
10 kirja ElementaryBaseSpan(56, 61) ['▁kirja']
11 pandud ElementaryBaseSpan(62, 68) ['▁pandud']
12 infost ElementaryBaseSpan(69, 75) ['▁inf', 'ost']
13 huvitab ElementaryBaseSpan(76, 83) ['▁huvitab']
14 vaid ElementaryBaseSpan(84, 88) ['▁vaid']
15 autorit ElementaryBaseSpan(89, 96) ['▁autorit']
16 v ElementaryBaseSpan(97, 98) ['▁v']
17 � ElementaryBaseSpan(98, 99) ['▁i']
18 i ElementaryBaseSpan(99, 100) ['▁i']
19 selle ElementaryBaseSpan(101, 106) ['▁selle']
20 l ElementaryBaseSpan(107, 108) ['▁l']
21 � ElementaryBaseSpan(108, 109) ['▁h']
2

In [41]:
morph_tagger._tokenize_with_bert(text_obj.text)

([(None, None, '<s>'),
  (0, 1, '▁('),
  (1, 9, 'Ameerika'),
  (9, 19, '▁teadlased'),
  (19, 22, '▁on'),
  (22, 31, '▁kindlaks'),
  (31, 38, '▁teinud'),
  (38, 39, ','),
  (39, 42, '▁et'),
  (42, 46, '▁99%'),
  (46, 51, '▁blog'),
  (51, 55, 'ides'),
  (55, 61, '▁kirja'),
  (61, 68, '▁pandud'),
  (68, 72, '▁inf'),
  (72, 75, 'ost'),
  (75, 83, '▁huvitab'),
  (83, 88, '▁vaid'),
  (88, 96, '▁autorit'),
  (96, 98, '▁v'),
  (98, 100, '▁i'),
  (100, 106, '▁selle'),
  (106, 108, '▁l'),
  (108, 110, '▁h'),
  (110, 113, 'iko'),
  (113, 115, 'nd'),
  (115, 119, 'lasi'),
  (119, 121, '.)'),
  (122, 130, '▁Tallinn'),
  (130, 131, '-'),
  (131, 136, 'Tartu'),
  (136, 137, '-'),
  (137, 145, 'Tallinna'),
  (145, 147, '▁p'),
  (147, 149, '▁e'),
  (149, 150, 'v'),
  (150, 153, '▁--'),
  (153, 160, '▁Tartus'),
  (160, 162, '▁l'),
  (162, 165, '▁pu'),
  (165, 169, 'pidu'),
  (169, 172, '▁ja'),
  (172, 180, '▁diplomi'),
  (180, 182, '▁k'),
  (182, 184, '▁t'),
  (184, 186, 'te'),
  (186, 194, '▁saamine'),

In [ ]:
group_words_by_bert_tokens(
    text_obj.words, map_words_to_bert_tokens(text_obj.words, morph_layer)
)

[([Span('Ning', [{'normalized_form': None}])],
  [Span('Ning', [{'bert_tokens': '▁Ning', 'morph_labels': 'J', 'probabilities': 0.99991}])]),
 ([Span('laste', [{'normalized_form': None}])],
  [Span(' laste', [{'bert_tokens': '▁laste', 'morph_labels': 'pl g_S', 'probabilities': 0.9996}])]),
 ([Span('kingitus', [{'normalized_form': None}])],
  [Span(' kingitus', [{'bert_tokens': '▁kingitus', 'morph_labels': 'sg n_S', 'probabilities': 0.99978}])]),
 ([Span(':', [{'normalized_form': None}])],
  [Span(':', [{'bert_tokens': ':', 'morph_labels': 'Z', 'probabilities': 0.99992}])]),
 ([Span('puust', [{'normalized_form': None}])],
  [Span(' puust', [{'bert_tokens': '▁puust', 'morph_labels': 'sg el_S', 'probabilities': 0.99941}])]),
 ([Span('juuksehari', [{'normalized_form': None}])],
  [Span(' juukse', [{'bert_tokens': '▁juukse', 'morph_labels': 'sg n_S', 'probabilities': 0.99975}]),
   Span('hari', [{'bert_tokens': 'hari', 'morph_labels': 'sg n_S', 'probabilities': 0.99975}])]),
 ([Span('(', [{'

In [ ]:
grouped_words = group_words_by_bert_tokens(
    text_obj.words, morph_layer, map_words_to_bert_tokens(text_obj.words, morph_layer)
)

In [ ]:
test_layer = Layer(
    name=morph_tagger.output_layer,
    attributes=morph_tagger.output_attributes,
    text_object=text_obj,
    ambiguous=True,
)
for sharing_words, shared_bert_tokens in grouped_words:
    if len(sharing_words) > 1:  # TODO: Add comment
        assert len(sharing_words) == len(shared_bert_tokens), (
            f"Sharing words and shared bert tokens aren't the same length\nsharing_words={len(sharing_words)}\nshared_bert_tokens={len(shared_bert_tokens)}"
        )
        for sharing_word_part, shared_bert_tokens_part in zip(
            sharing_words, shared_bert_tokens
        ):
            new_span = sharing_word_part.base_span
            annotation = rewriter_decorator(
                text_obj, [sharing_word_part], [shared_bert_tokens_part]
            )
            if annotation is not None:
                test_layer.add_annotation(new_span, annotation)
    else:
        new_span = [sp.base_span for sp in sharing_words][0]
        annotation = rewriter_decorator(text_obj, sharing_words, shared_bert_tokens)
        if annotation is not None:
            test_layer.add_annotation(new_span, annotation)

TypeError: 'float' object is not iterable

In [111]:
test_layer

Layer(name='bert_morph_tagging', attributes=('bert_tokens', 'morph_labels', 'probabilities'), spans=SL[Span('Ning', [{'bert_tokens': ['▁Ning'], 'morph_labels': ['J', 'o_V', 'D'], 'probabilities': [0.99991, 0.0, 0.0]}]),
Span('laste', [{'bert_tokens': ['▁laste'], 'morph_labels': ['pl g_S', 'sg g_S', 'pl g_A'], 'probabilities': [0.9996, 6e-05, 3e-05]}]),
Span('kingitus', [{'bert_tokens': ['▁kingitus'], 'morph_labels': ['sg n_S', 'sg in_S', 'sg p_S'], 'probabilities': [0.99978, 3e-05, 1e-05]}]),
Span(':', [{'bert_tokens': [':'], 'morph_labels': ['Z', 'J', 'b_V'], 'probabilities': [0.99992, 2e-05, 0.0]}]),
Span('puust', [{'bert_tokens': ['▁puust'], 'morph_labels': ['sg el_S', 'sg p_S', 'sg el_H'], 'probabilities': [0.99941, 0.00012, 4e-05]}]),
Span('juuksehari', [{'bert_tokens': ['▁juukse', 'hari'], 'morph_labels': ['sg n_S', 'sg in_S', 'sg p_S'], 'probabilities': [0.99975, 4e-05, 3e-05]}]),
Span('(', [{'bert_tokens': ['▁('], 'morph_labels': ['Z', 'J', 'A'], 'probabilities': [0.99987, 5e-05, 0.0]}]),
Span('millist', [{'bert_tokens': ['mil', 'list'], 'morph_labels': ['sg p_P', 'sg el_P', 'sg p_A'], 'probabilities': [0.99783, 0.00098, 0.0001]}]),
Span('ma', [{'bert_tokens': ['▁ma'], 'morph_labels': ['sg n_P', 'pl n_P', 'sg g_P'], 'probabilities': [0.99983, 1e-05, 1e-05]}]),
Span('olin', [{'bert_tokens': ['▁olin'], 'morph_labels': ['sin_V', 'sime_V', 'n_V'], 'probabilities': [0.99878, 9e-05, 8e-05]}]),
Span('ammu', [{'bert_tokens': ['▁ammu'], 'morph_labels': ['D', 'A', 'sg g_S'], 'probabilities': [0.99989, 1e-05, 0.0]}]),
Span('endale', [{'bert_tokens': ['▁endale'], 'morph_labels': ['sg all_P', 'pl all_P', 'sg all_H'], 'probabilities': [0.9998, 2e-05, 1e-05]}]),
Span('soovinud', [{'bert_tokens': ['▁soovinud'], 'morph_labels': ['nud_V', 'pl n_S', 'A'], 'probabilities': [0.99973, 2e-05, 2e-05]}]),
Span(')', [{'bert_tokens': [')'], 'morph_labels': ['Z', 'J', 'A'], 'probabilities': [0.99995, 0.0, 0.0]}]),
Span(',', [{'bert_tokens': [','], 'morph_labels': ['Z', 'J', 'A'], 'probabilities': [0.99995, 0.0, 0.0]}]),
Span('tulbid', [{'bert_tokens': ['▁tul', 'bid'], 'morph_labels': ['pl n_S', 'pl n_A', 'pl n_H'], 'probabilities': [0.99949, 0.00012, 5e-05]}]),
Span('ja', [{'bert_tokens': ['▁ja'], 'morph_labels': ['J', 'o_V', 'D'], 'probabilities': [0.99991, 0.0, 0.0]}]),
Span('karp', [{'bert_tokens': ['▁karp'], 'morph_labels': ['sg n_S', 'sg in_S', 'pl g_S'], 'probabilities': [0.99976, 2e-05, 2e-05]}]),
Span('Rafaellot', [{'bert_tokens': ['▁Ra', 'fa', 'ell', 'ot'], 'morph_labels': ['sg p_H', 'sg n_H', 'pl n_H'], 'probabilities': [0.78436, 0.1822, 0.00609]}]),
Span('.', [{'bert_tokens': ['.'], 'morph_labels': ['Z', 'J', 'sg n_H'], 'probabilities': [0.99996, 0.0, 0.0]}])])

In [ ]:
bert_tokens = group_words_by_bert_tokens(
    text_obj.words, map_words_to_bert_tokens(text_obj.words, morph_layer)
)[13][1]
words = group_words_by_bert_tokens(
    text_obj.words, map_words_to_bert_tokens(text_obj.words, morph_layer)
)[13][0]
new_bert_tokens = []
token_index = 0

for word in words:
    # Ensure there is a corresponding BERT token
    if token_index < len(bert_tokens):
        bert_token = bert_tokens[token_index]
        print(bert_token)
        remaining_token = bert_token["bert_tokens"][0]  # Start with the full BERT token
        current_start = bert_token.start

        # While there is still part of the BERT token remaining to split
        while remaining_token:
            # Calculate the overlap between the word and the remaining BERT token
            word_length = word.end - word.start
            sub_token = remaining_token[
                :word_length
            ]  # Take the part that fits the current word
            remaining_token = remaining_token[
                word_length:
            ]  # The rest of the BERT token
            annotation_count = len(bert_token["morph_labels"])

            # Create a new sub-token with the same labels and probabilities and wrap it in Span
            new_sub_token_span = estnltk.Span(
                estnltk.ElementaryBaseSpan(
                    word.start, min(word.end, current_start + len(sub_token))
                ),
                morph_layer,
            )
            for i in range(annotation_count):
                new_sub_token_span.add_annotation(
                    {
                        "bert_tokens": sub_token,
                        "morph_labels": bert_token["morph_labels"][
                            i
                        ],  # Keep the original labels
                        "probabilities": bert_token["probabilities"][
                            i
                        ],  # Keep the original probabilities
                    }
                )
            new_bert_tokens.append(new_sub_token_span)
            current_start += len(sub_token)

            # If there is no remaining part of the token, move to the next word
            if not remaining_token:
                token_index += 1
                break

            # Move to the next word span
            word = next((w for w in words if w.start >= new_sub_token_span.end), None)
            if word is None:
                break  # No more words left

Span('),', [{'bert_tokens': '),', 'morph_labels': 'Z', 'probabilities': 0.99995}, {'bert_tokens': '),', 'morph_labels': 'J', 'probabilities': 0.0}, {'bert_tokens': '),', 'morph_labels': 'A', 'probabilities': 0.0}])


In [36]:
new_bert_tokens

[Span(')', [{'bert_tokens': ')', 'morph_labels': 'Z', 'probabilities': 0.99995}, {'bert_tokens': ')', 'morph_labels': 'J', 'probabilities': 0.0}, {'bert_tokens': ')', 'morph_labels': 'A', 'probabilities': 0.0}]),
 Span(',', [{'bert_tokens': ',', 'morph_labels': 'Z', 'probabilities': 0.99995}, {'bert_tokens': ',', 'morph_labels': 'J', 'probabilities': 0.0}, {'bert_tokens': ',', 'morph_labels': 'A', 'probabilities': 0.0}])]

In [40]:
for span in text_obj.words.spans:
    if span.normalized_form[0]:
        print(span.normalized_form[0])
    else:
        print(span.text)

Ennustaja
,
ma
pean
tarvilikuks
teid
informeerida
selles
,
et
sellest
"
lugejatrombsmaarikapiitertupolev
brittnolooginoff
"
esinen
ma
ainult
ühe
korra
,
s.t.
.
Maarikana
.
Teised
pole
kunagi
minu
aliased
olnud
.
Rahu
sulle
,
ära
üle
pinguta
.


In [ ]:
def rewrite_layer(self, text_obj, bert_tokens_layer, words_layer):
    """
    Combines tokens progressively to match the length and structure of the provided words list.
    Handles cases where tokens need to be recombined or split to match the words.
    """
    tokens = [token[0] for token in bert_tokens_layer.bert_tokens]
    words = [
        span.normalized_form[0] if span.normalized_form[0] is not None else span.text
        for span in words_layer.spans
    ]

    new_morph_layer = Layer(
        name=self.output_layer,
        attributes=self.output_attributes,
        text_object=text_obj,
        ambiguous=True,
    )

    token_buffer = ""
    collected_tokens = []
    word_index = 0
    token_index = 0
    token_start_index = 0
    next_token = True

    while token_index < len(tokens) and word_index < len(words):
        span = bert_tokens_layer.spans[token_index]

        # Clean the token by removing the leading '▁' if present
        clean_token = tokens[token_index].strip("▁")

        # If this is the first token in the word, capture the span start
        if token_buffer == "":
            span_start = span.start
            token_start_index = token_index

        # Accumulate token to the buffer
        if next_token:
            token_buffer += clean_token
            collected_tokens.append(tokens[token_index])

        # Case 1: Exact match with the current word
        if token_buffer == words[word_index]:
            span_end = span.end
            span_tuple = (span_start, span_end)

            for label, prob in zip(list(span.morph_labels), list(span.probabilities)):
                annotation = {
                    "bert_tokens": collected_tokens,
                    "morph_labels": label,
                    "probabilities": prob,
                }
                new_morph_layer.add_annotation(span_tuple, annotation)

            # Reset the buffer and collected tokens for the next word
            token_buffer = ""
            collected_tokens = []
            word_index += 1  # Move to the next word
            token_index += 1  # Move to the next token
            next_token = True

        # Case 2: Token buffer length exceeds the current word length, try splitting
        elif len(token_buffer) > len(words[word_index]):
            split_length = len(words[word_index])  # Get the length of the current word
            if token_buffer[:split_length] == words[word_index]:
                span_end = words_layer.spans[word_index].end
                span_tuple = (span_start, span_end)

                for label, prob in zip(
                    list(span.morph_labels), list(span.probabilities)
                ):
                    annotation = {
                        "bert_tokens": collected_tokens,
                        "morph_labels": label,
                        "probabilities": prob,
                    }
                    new_morph_layer.add_annotation(span_tuple, annotation)

                token_buffer = token_buffer[
                    split_length:
                ]  # Keep the remainder for the next word
                span_start = span_end
                word_index += 1  # Move to the next word
                next_token = False
            else:
                # Case 3: No match, move to the next token and continue combining
                token_index += 1
                next_token = True

        # Case 4: Buffer is still less than the current word, move to the next token to keep accumulating
        else:
            token_index += 1
            next_token = True

    # If there are any remaining tokens after the loop, add them as the final word
    if token_buffer and word_index < len(words):
        for label, prob in zip(list(span.morph_labels), list(span.probabilities)):
            annotation = {
                "bert_tokens": collected_tokens,
                "morph_labels": label,
                "probabilities": prob,
            }
            new_morph_layer.add_annotation(span_tuple, annotation)

    return new_morph_layer


rewrite_layer(morph_tagger, text_obj, morph_layer, text_obj.words)

Layer(name='bert_morph_tagging', attributes=('bert_tokens', 'morph_labels', 'probabilities'), spans=SL[Span('Ennustaja', [{'bert_tokens': ['▁Enn', 'ustaja'], 'morph_labels': 'sg n_S', 'probabilities': 0.99958}]),
Span(',', [{'bert_tokens': [','], 'morph_labels': 'Z', 'probabilities': 0.99996}]),
Span(' ma', [{'bert_tokens': ['▁ma'], 'morph_labels': 'sg n_P', 'probabilities': 0.99986}]),
Span(' pean', [{'bert_tokens': ['▁pean'], 'morph_labels': 'n_V', 'probabilities': 0.99952}]),
Span(' tarvilikuks', [{'bert_tokens': ['▁tarv', 'il', 'ikuks'], 'morph_labels': 'sg tr_A', 'probabilities': 0.99894}]),
Span(' teid', [{'bert_tokens': ['▁teid'], 'morph_labels': 'pl p_P', 'probabilities': 0.99611}]),
Span(' informeerida', [{'bert_tokens': ['▁informeer', 'ida'], 'morph_labels': 'da_V', 'probabilities': 0.99987}]),
Span(' selles', [{'bert_tokens': ['▁selles'], 'morph_labels': 'sg in_P', 'probabilities': 0.99934}]),
Span(',', [{'bert_tokens': [','], 'morph_labels': 'Z', 'probabilities': 0.99997}]),
Span(' et', [{'bert_tokens': ['▁et'], 'morph_labels': 'J', 'probabilities': 0.99991}]),
Span(' sellest', [{'bert_tokens': ['▁sellest'], 'morph_labels': 'sg el_P', 'probabilities': 0.99917}]),
Span(' "', [{'bert_tokens': ['▁"'], 'morph_labels': 'Z', 'probabilities': 0.99995}]),
Span('lugejatrombsmaarikapiitertupolev', [{'bert_tokens': ['lu', 'geja', 't', 'ro', 'mb', 's', 'maar', 'ika', 'pii', 'ter', 'tu', 'pole', 'v'], 'morph_labels': 'sg n_A', 'probabilities': 0.99668}]),
Span(' brittnolooginoff', [{'bert_tokens': ['▁britt', 'n', 'oloo', 'gin', 'off'], 'morph_labels': 'sg n_S', 'probabilities': 0.99961}]),
Span('"', [{'bert_tokens': ['"'], 'morph_labels': 'Z', 'probabilities': 0.99994}]),
Span(' esinen', [{'bert_tokens': ['▁esin', 'en'], 'morph_labels': 'n_V', 'probabilities': 0.99936}]),
Span(' ma', [{'bert_tokens': ['▁ma'], 'morph_labels': 'sg n_P', 'probabilities': 0.99986}]),
Span(' ainult', [{'bert_tokens': ['▁ainult'], 'morph_labels': 'D', 'probabilities': 0.9999}]),
Span(' ühe', [{'bert_tokens': ['▁ühe'], 'morph_labels': 'sg g_P', 'probabilities': 0.99935}]),
Span(' korra', [{'bert_tokens': ['▁korra'], 'morph_labels': 'sg g_S', 'probabilities': 0.99968}]),
Span(',', [{'bert_tokens': [','], 'morph_labels': 'Z', 'probabilities': 0.99997}]),
Span(' s.t.', [{'bert_tokens': ['▁s', '.', 't', '.'], 'morph_labels': 'Z', 'probabilities': 0.99956}]),
Span(' .', [{'bert_tokens': ['▁.'], 'morph_labels': 'Z', 'probabilities': 0.99996}, {'bert_tokens': ['▁Maar', 'ika', '-', 'na', '.', '▁Teised', '▁pole', '▁kunagi', '▁minu', '▁al', 'ia', 'sed', '▁olnud', '.', '▁Rahu', '▁sulle', ',', '▁ära', '▁üle', '▁pingu', 'ta', '.'], 'morph_labels': 'Z', 'probabilities': 0.99997}])])

In [30]:
top_n_predictions, tokens_with_spans

NameError: name 'tokens_with_spans' is not defined

In [31]:
morph_layer

Layer(name='bert_morph_tagging', attributes=('bert_tokens', 'morph_labels', 'probabilities'), spans=SL[Span('Ning', [{'bert_tokens': '▁Ning', 'morph_labels': 'J', 'probabilities': 0.99991}, {'bert_tokens': '▁Ning', 'morph_labels': 'o_V', 'probabilities': 0.0}, {'bert_tokens': '▁Ning', 'morph_labels': 'D', 'probabilities': 0.0}]),
Span(' laste', [{'bert_tokens': '▁laste', 'morph_labels': 'pl g_S', 'probabilities': 0.9996}, {'bert_tokens': '▁laste', 'morph_labels': 'sg g_S', 'probabilities': 6e-05}, {'bert_tokens': '▁laste', 'morph_labels': 'pl g_A', 'probabilities': 3e-05}]),
Span(' kingitus', [{'bert_tokens': '▁kingitus', 'morph_labels': 'sg n_S', 'probabilities': 0.99978}, {'bert_tokens': '▁kingitus', 'morph_labels': 'sg in_S', 'probabilities': 3e-05}, {'bert_tokens': '▁kingitus', 'morph_labels': 'sg p_S', 'probabilities': 1e-05}]),
Span(':', [{'bert_tokens': ':', 'morph_labels': 'Z', 'probabilities': 0.99992}, {'bert_tokens': ':', 'morph_labels': 'J', 'probabilities': 2e-05}, {'bert_tokens': ':', 'morph_labels': 'b_V', 'probabilities': 0.0}]),
Span(' puust', [{'bert_tokens': '▁puust', 'morph_labels': 'sg el_S', 'probabilities': 0.99941}, {'bert_tokens': '▁puust', 'morph_labels': 'sg p_S', 'probabilities': 0.00012}, {'bert_tokens': '▁puust', 'morph_labels': 'sg el_H', 'probabilities': 4e-05}]),
Span(' juukse', [{'bert_tokens': '▁juukse', 'morph_labels': 'sg n_S', 'probabilities': 0.99975}, {'bert_tokens': '▁juukse', 'morph_labels': 'sg in_S', 'probabilities': 4e-05}, {'bert_tokens': '▁juukse', 'morph_labels': 'sg p_S', 'probabilities': 3e-05}]),
Span('hari', [{'bert_tokens': 'hari', 'morph_labels': 'sg n_S', 'probabilities': 0.99975}, {'bert_tokens': 'hari', 'morph_labels': 'sg in_S', 'probabilities': 4e-05}, {'bert_tokens': 'hari', 'morph_labels': 'sg p_S', 'probabilities': 2e-05}]),
Span(' (', [{'bert_tokens': '▁(', 'morph_labels': 'Z', 'probabilities': 0.99987}, {'bert_tokens': '▁(', 'morph_labels': 'J', 'probabilities': 5e-05}, {'bert_tokens': '▁(', 'morph_labels': 'A', 'probabilities': 0.0}]),
Span('mil', [{'bert_tokens': 'mil', 'morph_labels': 'sg p_P', 'probabilities': 0.99783}, {'bert_tokens': 'mil', 'morph_labels': 'sg el_P', 'probabilities': 0.00098}, {'bert_tokens': 'mil', 'morph_labels': 'sg p_A', 'probabilities': 0.0001}]),
Span('list', [{'bert_tokens': 'list', 'morph_labels': 'sg p_P', 'probabilities': 0.99854}, {'bert_tokens': 'list', 'morph_labels': 'sg el_P', 'probabilities': 0.00043}, {'bert_tokens': 'list', 'morph_labels': 'sg p_A', 'probabilities': 0.0001}]),
Span(' ma', [{'bert_tokens': '▁ma', 'morph_labels': 'sg n_P', 'probabilities': 0.99983}, {'bert_tokens': '▁ma', 'morph_labels': 'pl n_P', 'probabilities': 1e-05}, {'bert_tokens': '▁ma', 'morph_labels': 'sg g_P', 'probabilities': 1e-05}]),
Span(' olin', [{'bert_tokens': '▁olin', 'morph_labels': 'sin_V', 'probabilities': 0.99878}, {'bert_tokens': '▁olin', 'morph_labels': 'sime_V', 'probabilities': 9e-05}, {'bert_tokens': '▁olin', 'morph_labels': 'n_V', 'probabilities': 8e-05}]),
Span(' ammu', [{'bert_tokens': '▁ammu', 'morph_labels': 'D', 'probabilities': 0.99989}, {'bert_tokens': '▁ammu', 'morph_labels': 'A', 'probabilities': 1e-05}, {'bert_tokens': '▁ammu', 'morph_labels': 'sg g_S', 'probabilities': 0.0}]),
Span(' endale', [{'bert_tokens': '▁endale', 'morph_labels': 'sg all_P', 'probabilities': 0.9998}, {'bert_tokens': '▁endale', 'morph_labels': 'pl all_P', 'probabilities': 2e-05}, {'bert_tokens': '▁endale', 'morph_labels': 'sg all_H', 'probabilities': 1e-05}]),
Span(' soovinud', [{'bert_tokens': '▁soovinud', 'morph_labels': 'nud_V', 'probabilities': 0.99973}, {'bert_tokens': '▁soovinud', 'morph_labels': 'pl n_S', 'probabilities': 2e-05}, {'bert_tokens': '▁soovinud', 'morph_labels': 'A', 'probabilities': 2e-05}]),
Span('),', [{'bert_tokens': '),', 'morph_labels': 'Z', 'probabilities': 0.99995}, {'bert_tokens': '),', 'morph_labels': 'J', 'probabilities': 0.0}, {'bert_tokens': '),', 'morph_labels': 'A', 'probabilities': 0.0}]),
Span(' tul', [{'bert_toke

In [94]:
# Apply the tagger to the text object
morph_tagger.make_layer(text_obj)

Layer(name='bert_morph_tagging', attributes=('bert_tokens', 'morph_labels', 'probabilities'), spans=SL[EnvelopingSpan(['Ning'], [{'bert_tokens': ['▁Ning'], 'morph_labels': 'J', 'probabilities': 0.99991}]),
EnvelopingSpan(['laste'], [{'bert_tokens': ['▁laste'], 'morph_labels': 'pl g_S', 'probabilities': 0.9996}]),
EnvelopingSpan(['kingitus'], [{'bert_tokens': ['▁kingitus'], 'morph_labels': 'sg n_S', 'probabilities': 0.99978}]),
EnvelopingSpan([':'], [{'bert_tokens': [':'], 'morph_labels': 'Z', 'probabilities': 0.99992}]),
EnvelopingSpan(['puust'], [{'bert_tokens': ['▁puust'], 'morph_labels': 'sg el_S', 'probabilities': 0.99941}]),
EnvelopingSpan(['juuksehari'], [{'bert_tokens': ['▁juukse', 'hari'], 'morph_labels': 'sg n_S', 'probabilities': 0.99975}]),
EnvelopingSpan(['('], [{'bert_tokens': ['▁('], 'morph_labels': 'Z', 'probabilities': 0.99987}]),
EnvelopingSpan(['millist'], [{'bert_tokens': ['mil', 'list'], 'morph_labels': 'sg p_P', 'probabilities': 0.99783}]),
EnvelopingSpan(['ma'], [{'bert_tokens': ['▁ma'], 'morph_labels': 'sg n_P', 'probabilities': 0.99983}]),
EnvelopingSpan(['olin'], [{'bert_tokens': ['▁olin'], 'morph_labels': 'sin_V', 'probabilities': 0.99878}]),
EnvelopingSpan(['ammu'], [{'bert_tokens': ['▁ammu'], 'morph_labels': 'D', 'probabilities': 0.99989}]),
EnvelopingSpan(['endale'], [{'bert_tokens': ['▁endale'], 'morph_labels': 'sg all_P', 'probabilities': 0.9998}]),
EnvelopingSpan(['soovinud'], [{'bert_tokens': ['▁soovinud'], 'morph_labels': 'nud_V', 'probabilities': 0.99973}]),
EnvelopingSpan([')', ','], [{'bert_tokens': ['),'], 'morph_labels': 'Z', 'probabilities': 0.99995}]),
EnvelopingSpan(['tulbid'], [{'bert_tokens': ['▁tul', 'bid'], 'morph_labels': 'pl n_S', 'probabilities': 0.99949}]),
EnvelopingSpan(['ja'], [{'bert_tokens': ['▁ja'], 'morph_labels': 'J', 'probabilities': 0.99991}]),
EnvelopingSpan(['karp'], [{'bert_tokens': ['▁karp'], 'morph_labels': 'sg n_S', 'probabilities': 0.99976}]),
EnvelopingSpan(['Rafaellot'], [{'bert_tokens': ['▁Ra', 'fa', 'ell', 'ot'], 'morph_labels': 'sg p_H', 'probabilities': 0.78436}]),
EnvelopingSpan(['.'], [{'bert_tokens': ['.'], 'morph_labels': 'Z', 'probabilities': 0.99996}])])

### Testing _(will be removed afterwards)_


In [ ]:
test_text = estnltk.Text(
    "A. H. Tammsaare oli eesti kirjanik, esseist, kultuurifilosoof ja tõlkija. Üksnes autorihüvitis oli 12 431 krooni."
)
test_text.tag_layer("morph_analysis")
test_text.morph_analysis;

In [ ]:
sentences = [s.text for s in test_text.sentences]

In [ ]:
# Predict tags
predictions, raw_outputs = model.predict(sentences, split_on_space=False)

# Output of predictions
print(predictions)

INFO:ner_model.py:1884:  Converting to features started.
[[{'A. H. Tammsaare': 'sg n_H'}, {'oli': 's_V'}, {'eesti': 'G'}, {'kirjanik': 'sg n_S'}, {',': 'Z'}, {'esseist': 'sg n_S'}, {',': 'Z'}, {'kultuurifilosoof': 'sg n_S'}, {'ja': 'J'}, {'tõlkija': 'sg n_S'}, {'.': 'Z'}], [{'Üksnes': 'D'}, {'autorihüvitis': 'sg n_S'}, {'oli': 's_V'}, {'12 431': '?_N'}, {'krooni': 'sg p_S'}, {'.': 'Z'}]]


In [ ]:
display(test_text.morph_analysis)

Layer(name='morph_analysis', attributes=('normalized_text', 'lemma', 'root', 'root_tokens', 'ending', 'clitic', 'form', 'partofspeech'), spans=SL[Span('A. H. Tammsaare', [{'normalized_text': 'A. H. Tammsaare', 'lemma': 'A. H. Tammsaare', 'root': 'A. _H. _Tamm_saare', 'root_tokens': ['A. ', 'H. ', 'Tamm', 'saare'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'H'}]),
Span('oli', [{'normalized_text': 'oli', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': 'i', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('eesti', [{'normalized_text': 'eesti', 'lemma': 'eesti', 'root': 'eesti', 'root_tokens': ['eesti'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'G'}]),
Span('kirjanik', [{'normalized_text': 'kirjanik', 'lemma': 'kirjanik', 'root': 'kirjanik', 'root_tokens': ['kirjanik'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('esseist', [{'normalized_text': 'esseist', 'lemma': 'essee', 'root': 'essee', 'root_tokens': ['essee'], 'ending': 'ist', 'clitic': '', 'form': 'pl el', 'partofspeech': 'S'}]),
Span(',', [{'normalized_text': ',', 'lemma': ',', 'root': ',', 'root_tokens': [','], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('kultuurifilosoof', [{'normalized_text': 'kultuurifilosoof', 'lemma': 'kultuurifilosoof', 'root': 'kultuuri_filosoof', 'root_tokens': ['kultuuri', 'filosoof'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('ja', [{'normalized_text': 'ja', 'lemma': 'ja', 'root': 'ja', 'root_tokens': ['ja'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'J'}]),
Span('tõlkija', [{'normalized_text': 'tõlkija', 'lemma': 'tõlkija', 'root': 'tõlkija', 'root_tokens': ['tõlkija'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}]),
Span('Üksnes', [{'normalized_text': 'Üksnes', 'lemma': 'üksnes', 'root': 'üksnes', 'root_tokens': ['üksnes'], 'ending': '0', 'clitic': '', 'form': '', 'partofspeech': 'D'}]),
Span('autorihüvitis', [{'normalized_text': 'autorihüvitis', 'lemma': 'autorihüvitis', 'root': 'autori_hüvitis', 'root_tokens': ['autori', 'hüvitis'], 'ending': '0', 'clitic': '', 'form': 'sg n', 'partofspeech': 'S'}]),
Span('oli', [{'normalized_text': 'oli', 'lemma': 'olema', 'root': 'ole', 'root_tokens': ['ole'], 'ending': 'i', 'clitic': '', 'form': 's', 'partofspeech': 'V'}]),
Span('12 431', [{'normalized_text': '12431', 'lemma': '12431', 'root': '12431', 'root_tokens': ['12431'], 'ending': '0', 'clitic': '', 'form': '?', 'partofspeech': 'N'}]),
Span('krooni', [{'normalized_text': 'krooni', 'lemma': 'kroon', 'root': 'kroon', 'root_tokens': ['kroon'], 'ending': '0', 'clitic': '', 'form': 'sg p', 'partofspeech': 'S'}]),
Span('.', [{'normalized_text': '.', 'lemma': '.', 'root': '.', 'root_tokens': ['.'], 'ending': '', 'clitic': '', 'form': '', 'partofspeech': 'Z'}])])

In [ ]:
predictions, raw_outputs = model.predict(
    [
        "Uuringutulemuste interpretatsioon jätab mitmed küsimused vastuseta.",
        "Kas tuba on soe või külm?",
    ]
)

# Output of predictions
print(predictions)

INFO:ner_model.py:1884:  Converting to features started.
[[{'Uuringutulemuste': 'pl g_S'}, {'interpretatsioon': 'sg n_S'}, {'jätab': 'b_V'}, {'mitmed': 'pl n_P'}, {'küsimused': 'pl n_S'}, {'vastuseta.': 'sg ab_S'}], [{'Kas': 'D'}, {'tuba': 'sg n_S'}, {'on': 'b_V'}, {'soe': 'sg n_A'}, {'või': 'J'}, {'külm?': 'sg n_A'}]]


In [ ]:
test_data = comparison_data[
    (comparison_data["source"] == "nc_8303_518571.json")
    & (comparison_data["sentence_id"] == 797)
]

In [ ]:
text_list = test_data.words.tolist()

In [ ]:
test_data

,sentence_id,words,form,pos,type,source,labels
1467181,797,Tegevusalad,pl n,S,periodicals,nc_8303_518571.json,pl n_S
1467182,797,:,,Z,periodicals,nc_8303_518571.json,Z
1467183,797,puit-,sg n,S,periodicals,nc_8303_518571.json,sg n_S
1467184,797,",",,Z,periodicals,nc_8303_518571.json,Z
1467185,797,plast-,sg n,S,periodicals,nc_8303_518571.json,sg n_S
...,...,...,...,...,...,...,...
1467249,797,",",,Z,periodicals,nc_8303_518571.json,Z
1467250,797,paljundustööd,pl n,S,periodicals,nc_8303_518571.json,pl n_S
1467251,797,",",,Z,periodicals,nc_8303_518571.json,Z
1467252,797,infoteenused,pl n,S,periodicals,nc_8303_518571.json,pl n_S


In [ ]:
inputs = model.tokenizer(" ".join(text_list), return_tensors="pt")

In [ ]:
len(inputs["input_ids"][0])

128

In [ ]:
for token_idx in range(len(inputs["input_ids"][0])):
    token = model.tokenizer.decode([inputs["input_ids"][0][token_idx]])
    print(f"Token: {token}")

Token: <s>
Token: Tegevus
Token: alad
Token: :
Token: puit
Token: -
Token: ,
Token: plast
Token: -
Token: ,
Token: klaas
Token: -
Token: ,
Token: kummi
Token: -
Token: ja
Token: metall
Token: it
Token: oodete
Token: tootmine
Token: ,
Token: toidu
Token: kaupade
Token: tootmine
Token: ,
Token: toidu
Token: kaupade
Token: jae
Token: -
Token: ja
Token: hul
Token: gika
Token: ub
Token: andus
Token: (
Token: v
Token: .
Token: a
Token: .
Token: litse
Token: ntse
Token: eri
Token: tavad
Token: kaubad
Token: )
Token: ,
Token: tööstus
Token: kaupade
Token: jae
Token: -
Token: ,
Token: hulgi
Token: -
Token: ja
Token: komisjon
Token: ika
Token: ub
Token: andus
Token: ,
Token: arvuti
Token: -
Token: ,
Token: video
Token: -
Token: ja
Token: hel
Token: isal
Token: vest
Token: iste
Token: palju
Token: ndamine
Token: ja
Token: laen
Token: utamine
Token: ,
Token: interj
Token: öör
Token: iku
Token: ju
Token: ndus
Token: ,
Token: kuju
Token: ndust
Token: ööd
Token: ,
Token: demonst
Token: ratsioonide
To

In [ ]:
text = estnltk.Text("  ".join(text_list))
text.tag_layer("clauses")
display(text)
clauses = [clause.text for clause in text.clauses]
for clause in clauses:
    print(len(clause))
print("Sum:", sum(len(clause) for clause in clauses))

Text(text='Tegevusalad  :  puit-  ,  plast-  ,  klaas-  ,  kummi-  ja  metallitoodete  tootmine  ,  toidukaupade  tootmine  ,  toidukaupade  jae-  ja  hulgikaubandus  (  v . a .  litsentseeritavad  kaubad  )  ,  tööstuskaupade  jae-  ,  hulgi-  ja  komisjonikaubandus  ,  arvuti-  ,  video-  ja  helisalvestiste  paljundamine  ja  laenutamine  ,  interjöörikujundus  ,  kujundustööd  ,  demonstratsioonide  ,  näituste  ,  nõupidamiste  ,  konverentside  ja  spordivõistluste  organiseerimine  ,  ärikontaktide  vahendamine  ,  transporditeenused  (  Eesti  Vabariigi  piires  )  ,  teeninduspunktid  ,  paljundustööd  ,  infoteenused  .')

2
61
5
5
Sum: 73


In [ ]:
for i in range(len(text_list)):
    print(text_list[i], text.words[i].text)

Tegevusalad Tegevusalad
: :
puit- puit-
, ,
plast- plast-
, ,
klaas- klaas-
, ,
kummi- kummi-
ja ja
metallitoodete metallitoodete
tootmine tootmine
, ,
toidukaupade toidukaupade
tootmine tootmine
, ,
toidukaupade toidukaupade
jae- jae-
ja ja
hulgikaubandus hulgikaubandus
( (
v . a . v . a .
litsentseeritavad litsentseeritavad
kaubad kaubad
) )
, ,
tööstuskaupade tööstuskaupade
jae- jae-
, ,
hulgi- hulgi-
ja ja
komisjonikaubandus komisjonikaubandus
, ,
arvuti- arvuti-
, ,
video- video-
ja ja
helisalvestiste helisalvestiste
paljundamine paljundamine
ja ja
laenutamine laenutamine
, ,
interjöörikujundus interjöörikujundus
, ,
kujundustööd kujundustööd
, ,
demonstratsioonide demonstratsioonide
, ,
näituste näituste
, ,
nõupidamiste nõupidamiste
, ,
konverentside konverentside
ja ja
spordivõistluste spordivõistluste
organiseerimine organiseerimine
, ,
ärikontaktide ärikontaktide
vahendamine vahendamine
, ,
transporditeenused transporditeenused
( (
Eesti Eesti
Vabariigi Vabariigi
piires piire

In [ ]:
predictions, raw_outputs = model.predict(clauses, split_on_space=False)
ner_labels_parts = list()
for prediction_part in predictions:
    ner_labels_part = [list(p.values())[0] for p in prediction_part]
    print(len(ner_labels_part), ner_labels_part)
    ner_labels_parts.append(ner_labels_part)

ner_labels = list(itertools.chain.from_iterable(ner_labels_parts))

2 ['pl n_S', 'Z']
61 ['sg n_S', 'Z', 'sg n_S', 'Z', 'sg n_S', 'Z', 'sg g_S', 'J', 'pl g_S', 'sg n_S', 'Z', 'pl g_S', 'sg n_S', 'Z', 'pl g_S', 'sg g_S', 'J', 'sg n_S', 'Z', 'pl g_S', 'sg g_S', 'Z', 'sg n_S', 'J', 'sg n_S', 'Z', 'sg g_S', 'Z', 'sg g_S', 'J', 'pl g_S', 'sg n_S', 'J', 'sg n_S', 'Z', 'sg n_S', 'Z', 'sg n_S', 'Z', 'pl g_S', 'Z', 'pl g_S', 'Z', 'pl g_S', 'Z', 'pl g_S', 'J', 'pl g_S', 'sg n_S', 'Z', 'pl g_S', 'sg n_S', 'Z', 'pl n_S', 'Z', 'pl n_S', 'Z', 'sg p_S', 'Z', 'pl n_S', 'Z']
5 ['Z', '?_Y', 'pl n_A', 'pl n_S', 'Z']
5 ['Z', 'sg g_H', 'sg g_S', 'K', 'Z']


In [ ]:
predictions, raw_outputs = model.predict([text.words.text], split_on_space=False)

# ner_words = [list(p.keys())[0] for p in predictions[0]]
ner_labels = [list(p.values())[0] for p in predictions[0]]

In [ ]:
len(ner_labels)

72

In [ ]:
predictions

[[{'Sisaldab': 'b_V'}, {':': 'Z'}],
 [{'dementsus': 'sg n_S'}, {':': 'Z'}],
 [{'-': 'Z'},
  {'CO-mürgistusest': 'sg el_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'tserebraalsest': 'sg el_A'},
  {'lipidoosist': 'sg el_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'epilepsiast': 'sg el_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'hepatolentikulaarsest': 'sg el_A'},
  {'degeneratsioonist': 'sg el_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'hüperkaltseemiast': 'sg el_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'kilpnäärmehaigustest': 'pl el_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'intoksikatsioonidest': 'pl el_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'sclerosis': 'sg n_S'},
  {"multiplex ' ist": 'sg n_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'neurosüüfilisest': 'sg el_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'niatsiinivaegusest': 'sg el_S'},
  {';': 'Z'},
  {'-': 'Z'},
  {'polyarteritis': 'sg in_S'},
  {"nodosa ' st": 'sg n_S'},
  {'-': 'Z'},
  {'süsteemsest': 'sg el_A'},
  {'lupus': 'sg n_S'},
  {'erythematosus': 'sg n_S'},
  {"'": 'Z'},
  {'est'

In [ ]:
len(ner_labels)

132

In [ ]:
text = estnltk.Text(
    'Paki tagaküljele - " Suitsetamine lühendab eluiga , " " Suitsetamine ummistab veresooni ning põhjustab südameinfarkte ja rabandusi , " " Suitsetamine tekitab surmavat kopsuvähki , " " Raseduse ajal suitsetamine kahjustab sinu last , " " Kaitse lapsi : ära sunni neid hingama tubakasuitsu , " " Arstilt või apteekrilt saad abi suitsetamisest loobumiseks , " " Suitsetamine tekitab kergesti sõltuvust , ära alusta , " " Suitsetamisest loobumine vähendab ohtu haigestuda surmavatesse südame- ja kopsuhaigustesse , " " Otsi abi suitsetamisest loobumiseks : küsi nõu oma perearstilt või lähimast apteegist . "'
)
# text = estnltk.Text("Mul on taskus kolm asja: telefon, rahakott ja võtmed.")
# text = estnltk.Text("Ta ei söö herneid, sest need ei maitse talle.")
text.tag_layer("clauses")
clauses = [clause.text for clause in text.clauses]

In [ ]:
text.clauses

Layer(name='clauses', attributes=('clause_type',), spans=SL[EnvelopingSpan(['231', 'Kirikuseadus', 'jaguneb', 'kahekümne', 'kaheksaks', 'peatükiks', ':'], [{'clause_type': 'regular'}]),
EnvelopingSpan(['1', ')', 'Õigest', 'kristlikust', 'õpetusest', ',', '2', ')', 'Jutlustest', 'ja', 'korrakohasest', 'jumalateenistusest', ',', '3', ')', 'Ristimisest', ',', '4', ')', 'Hädaristimisest', ',', '5', ')', 'Lapsevoodi-naiste', 'kirikuskäimisest', ',', '232', '6', ')', 'Pihtimisest', 'ja', 'pattude', 'andeksandmisest', ',', '7', ')', 'Salajasest', 'pihtimisest', ',', '8', ')', 'Üldisest', 'pihtimisest', ',', '9', ')', 'Avalikust', 'pihtimisest', 'ja', 'kiriklikust', 'patukahetsusest', ',', '10', ')', 'Kirikuvandest', ',', '11', ')', 'Armulauast', ',', '12', ')', 'Üldisest', 'palvest', 'ja', 'litaaniast', ',', '13', ')', 'Kirikulauludest', ',', '14', ')', 'Tähtsatest', 'pühadest', ',', '15', ')', 'Kihlusest', 'ja', 'abielust', ',', '16', ')', 'Kihluse', 'ja', 'abielu', 'lahutamisest', ',', '17', ')', 'Haigete', 'ja', 'vangide', 'külastamisest', ',', '18', ')', 'Kristlikust', 'matusest', ',', '19', ')', 'Pastoriametist', ',', '20', ')', 'Piiskoppide', 'valimisest', ',', '21', ')', 'Kuidas', 'valitud', 'piiskoppi', 'ordineerida', ',', '22', ')', 'Kuidas', 'pastorit', 'ordineerida', ',', '23', ')', 'Kuidas', 'korrakohaselt', 'ametissekutsutud', 'pastorit', 'koguduses', 'ametisse', 'panna', ',', '24', ')', 'Piiskoppidest', ',', 'superintendentidest', ',', 'praostidest', ',', 'pastoritest', ',', 'kaplanitest', 'ja', 'teistest', 'kirikuteenijatest', ',', '25', ')', 'Sinoditest', 'ehk', 'pastorite', 'kokkutulekutest', ',', '26', ')', 'Kirikutest', ',', 'nende', 'sisseseadest', 'ja', 'omandist', ',', '27', ')', 'Kuidas', 'peab', 'algama', 'ja', 'toimuma', 'jumalateenistus', 'vastvalminud', 'kirikus', ',', '28', ')', 'Hospidalidest', '.'], [{'clause_type': 'regular'}]),
EnvelopingSpan(['(', 'Kuuenädalaste', ')'], [{'clause_type': 'embedded'}])])

In [ ]:
clauses_text = [" ".join(clause) for clause in clauses]
full_text = " ".join(clauses_text)
diff_text = estnltk.Text(full_text)
diff_text.tag_layer("clauses")
diff_clauses = [clause.text for clause in diff_text.clauses]

NameError: name 'clauses' is not defined

In [ ]:
diff_text.clauses

Layer(name='clauses', attributes=('clause_type',), spans=SL[EnvelopingSpan(['Paki', 'tagaküljele', '-', '"', 'Suitsetamine', 'lühendab', 'eluiga', ','], [{'clause_type': 'regular'}]),
EnvelopingSpan(['"', '"', 'Suitsetamine', 'ummistab', 'veresooni', 'ning'], [{'clause_type': 'regular'}]),
EnvelopingSpan(['põhjustab', 'südameinfarkte', 'ja', 'rabandusi', ','], [{'clause_type': 'regular'}]),
EnvelopingSpan(['"', '"', 'Suitsetamine', 'tekitab', 'surmavat', 'kopsuvähki', ','], [{'clause_type': 'regular'}]),
EnvelopingSpan(['"', '"', 'Raseduse', 'ajal', 'suitsetamine', 'kahjustab', 'sinu', 'last', ','], [{'clause_type': 'regular'}]),
EnvelopingSpan(['"', '"', 'Kaitse', 'lapsi', ':'], [{'clause_type': 'regular'}]),
EnvelopingSpan(['ära', 'sunni', 'neid', 'hingama', 'tubakasuitsu', ','], [{'clause_type': 'regular'}]),
EnvelopingSpan(['"', '"', 'Arstilt', 'või', 'apteekrilt', 'saad', 'abi', 'suitsetamisest', 'loobumiseks', ','], [{'clause_type': 'regular'}]),
EnvelopingSpan(['"', '"', 'Suitsetamine', 'tekitab', 'kergesti', 'sõltuvust', ','], [{'clause_type': 'regular'}]),
EnvelopingSpan(['ära', 'alusta', ','], [{'clause_type': 'regular'}]),
EnvelopingSpan(['"', '"', 'Suitsetamisest', 'loobumine', 'vähendab', 'ohtu', 'haigestuda', 'surmavatesse', 'südame-', 'ja', 'kopsuhaigustesse', ','], [{'clause_type': 'regular'}]),
EnvelopingSpan(['"', '"', 'Otsi', 'abi', 'suitsetamisest', 'loobumiseks', ':'], [{'clause_type': 'regular'}]),
EnvelopingSpan(['küsi', 'nõu', 'oma', 'perearstilt', 'või', 'lähimast', 'apteegist', '.', '"'], [{'clause_type': 'regular'}])])

In [ ]:
for clause in text.clauses:
    display(clause.morph_analysis)

In [ ]:
with open(
    file="./differences/_nc_255_27987.json__ann_diffs.txt", mode="r", encoding="UTF-8"
) as in_f:
    for i, line in enumerate(in_f):
        if i > 20:
            break
        line = line.strip()
        print(line)


::nc_255_27987.json::202969

... {Kahtlused}  Ühendriikide majanduse kiire kasvutempo jätkumise suhtes ho...
--- MODIFIED --------------------------------------------------
morph_analysis_flat        ['H', 'pl n']
bert_vabamorf_tags_flat    ['S', 'pl n']


::nc_255_27987.json::202970

...ad dollarit surve all , Euroopa ühisraha murdis esmaspäeval  {läbi}  0,900 dollari piiri viimase nelja kuu kõrgeima tasemeni , v...
--- MODIFIED --------------------------------------------------
morph_analysis_flat        ['K', '']
bert_vabamorf_tags_flat    ['D', '']


::nc_255_27987.json::202971


In [ ]:
!python pick_randomly_from_diffs.py differences__ann_diffs_.txt 100 -e

INFO:pick_randomly_from_diffs.py:66: Collecting difference indexes ...
ERROR:pick_randomly_from_diffs.py:77: (!) No differences found from the given ann_diffs_file. Invalid file, perhaps?


### Testing v2


In [ ]:
in_dir = "_plain_texts_json"
jsons = os.listdir(in_dir)
train_data = pd.read_csv("./model_data.csv", keep_default_na=False)
morph_tagger_v2 = BertMorphTagger("./NER_mudel_v2/")
warnings.filterwarnings("ignore", module="bert_tokens_to_words_rewriter")

In [ ]:
used_sources = set(train_data["source"].unique())
unused_jsons = [json_file for json_file in jsons if json_file not in used_sources]

In [ ]:
print(len(jsons))
print(len(used_sources))
print(
    f"{len(jsons)} - {len(used_sources)} = {len(jsons) - len(used_sources)}, actual: {len(unused_jsons)}"
)

18486
1116
18486 - 1116 = 17370, actual: 17370


In [ ]:
# TODO: lahendada Vabamorfi ja BERT taggeri span'ide erinevused
# ['nc_17153_741265.json', 'nc_17164_741348.json']

In [ ]:
unused_jsons.remove("nc_17153_741265.json")
unused_jsons.remove("nc_17164_741348.json")

In [ ]:
counter = 0
failed_jsons = ["nc_17153_741265.json", "nc_17164_741348.json"]
for j in tqdm(unused_jsons):
    with open(os.path.join("_diff_morph_texts_json_v2_complete", j), "r") as f:
        t = json.load(f)
        if t["layers"][1]["name"] != "bert_morph_tagging":
            counter += 1
            failed_jsons.append(j)

print(counter)
print(failed_jsons)

  0%|          | 0/17368 [00:00<?, ?it/s]


UnicodeDecodeError: 'charmap' codec can't decode byte 0x81 in position 5181: character maps to <undefined>

In [ ]:
NotebookFunctions.create_json_file_by_file_enc2017(
    unused_jsons,
    in_dir,
    "_diff_morph_texts_json_v2_complete",
    "enc2017",
    True,
    bert_morph_tagger=morph_tagger_v2,
    ignore_errors=False,
    replace_files=False,
)

Beginning to morphologically tag file by file


100%|██████████| 17368/17368 [00:03<00:00, 4919.72it/s]

Morphological tagging completed successfully


##### Splitting data to sets<!-- #### Andmete jagamine hulkadesse -->


In [ ]:
# Reading CSV file
data = pd.read_csv("model_data.csv", keep_default_na=False)

Grouping data by filename to preserve the integrity of texts<!-- Andmete grupeerimine failinimede kaupa, et säilitada tekstide terviklikkus -->


In [ ]:
grouped = data.groupby("source")

groups = list(grouped.groups.keys())
train_groups, test_groups = sk.model_selection.train_test_split(
    groups, test_size=0.2, random_state=42
)


def filter_by_group(df, groups):
    return df[df["source"].isin(groups)]


# Splitting dataframe
train_df = filter_by_group(data, train_groups)
test_df = filter_by_group(data, test_groups)

Removing unnecessary columns for the model<!-- Mudelile ebavajalike veergude eemaldamine -->


In [ ]:
train_df = train_df.drop(labels=["type", "source"], axis=1)
test_df = test_df.drop(labels=["type", "source"], axis=1)
# display(train_df)
# display(test_df)

In [ ]:
with open("unique_labels.json", "r") as f:
    unique_labels = json.load(f)

In [ ]:
from bert_morph_tagger_notebook_functions import NotebookFunctions

with open("unique_labels.json", "r") as f:
    unique_labels = json.load(f)

model = NotebookFunctions.initialize_model("NER_mudel_v2", unique_labels, False)
result, model_outputs, preds_list = model.eval_model(test_df)
print(f"Evaluation Loss:{result['eval_loss']:.4f}")
print(f"Precision: \t{result['precision']:.4f}")
print(f"Recall: \t{result['recall']:.4f}")
print(f"F1 Score: \t{result['f1_score']:.4f}")

Running Evaluation:   0%|          | 0/4 [00:00<?, ?it/s]e:\Anaconda3\envs\gpulocal\Lib\site-packages\simpletransformers\ner\ner_model.py:1303: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
Running Evaluation: 100%|██████████| 4/4 [00:03<00:00,  1.30it/s]


Evaluation Loss:1.1393
Precision: 	0.8878
Recall: 	0.8134
F1 Score: 	0.8489


| Metric          | Bert_morph_v2 | Bert_morph_v1 |
| --------------- | ------------- | ------------- |
| Evaluation Loss | 1.1393        | 0.2393        |
| Precision       | 0.8878        | 0.9545        |
| Recall          | 0.8134        | 0.9534        |
| F1 Score        | 0.8489        | 0.9540        |
